# Hardware-Aware Gateway — Colab bootstrap

Runs the CUDA half of the project on a free T4.

**Set the runtime first:** Runtime → Change runtime type → T4 GPU. The first
cell fails deliberately if you forget, because everything below it would
otherwise skip silently and look like it passed.


In [ ]:
import torch

assert torch.cuda.is_available(), (
    'No CUDA device. Runtime -> Change runtime type -> T4 GPU, then rerun.'
)
print(torch.cuda.get_device_name(0))
print('torch', torch.__version__)
import triton; print('triton', triton.__version__)


## 1. Get the code

Triton already ships with Colab's torch build, so there is nothing to install
for the kernels themselves.


In [ ]:
!git clone -q https://github.com/Rahu378/hardware-aware-gateway.git
%cd hardware-aware-gateway
!pip install -q -r requirements-e2e.txt
import sys; sys.path.insert(0, 'src')


## 2. Correctness before speed

The CUDA half of the suite was written on a Mac and has never executed on an
NVIDIA device. This cell is the first real test of it. Expect to fix
something here — that is the point of running it.


In [ ]:
!python -m pytest -q


## 3. Baseline profile — find the traffic jam

Profile *before* changing anything. `nsys` works fine here despite the common
claim that notebooks cannot profile; it is `ncu` that needs a VM you control.

Read `cuda_gpu_kern_sum` first. If elementwise kernels outrank the GEMMs, the
model is memory-bound and fusion is the right lever.


In [ ]:
!apt-get -qq install -y nsight-systems-cli 2>/dev/null | tail -1
!nsys profile --trace=cuda,nvtx,osrt --cuda-memory-usage=true \
    --force-overwrite=true -o profiles/baseline_t4 \
    python -m hag.bench_e2e --model Qwen/Qwen2.5-1.5B --prompt-tokens 512 --new-tokens 64


In [ ]:
!nsys stats --report cuda_gpu_kern_sum profiles/baseline_t4.nsys-rep | head -30


## 4. Op-level sweep

Writes `results/ops_nvidia-t4_fp16.json`. Note the launch-floor line at the
top — on a T4 it is far lower than on Apple silicon, so more of the decode
regime becomes measurable.


In [ ]:
!python -m hag.bench_ops --backend cuda --dtype fp16


## 5. End-to-end

The number that decides whether the kernel work mattered. A large op-level
speedup on an op that occupies 4% of the forward pass moves this by 4%.


In [ ]:
!python -m hag.bench_e2e --model Qwen/Qwen2.5-1.5B --prompt-tokens 512 --new-tokens 128


## 6. Regenerate the README tables and save the results

Download `results/` and `profiles/` and commit them from your machine, so the
repo's history shows the runs rather than just the code.


In [ ]:
!python -m hag.report
!sed -n '/BENCH:BEGIN/,/BENCH:END/p' README.md


In [ ]:
from google.colab import files

!zip -qr artifacts.zip results profiles
files.download('artifacts.zip')


---

### Next: Nsight Compute counters

`ncu` will fail on Colab with `ERR_NVGPUCTRPERM` — the runtime does not grant
performance-counter access. That is the one step that needs a GCP or Azure VM
on signup credits. See `scripts/profile_ncu.sh` for the module-parameter fix.
